# 01 - Data Understanding in **SQL** (DuckDB)

Same steps as the pandas notebook, in SQL. Select the **`.venv`** kernel, run top to bottom.
`run("...")` executes a query and shows the result as a table.

In [ ]:
import duckdb
from pathlib import Path

# robust path: works whether the working dir is the repo root OR notebooks/
DATA_DIR = next(p for p in [Path('data'), Path('../data')] if p.exists()).resolve()
con = duckdb.connect(str(DATA_DIR / 'credit.duckdb'))

# Build tables if a fresh clone hasn't yet. Instant if they already exist.
acc = (DATA_DIR / 'accepted_2007_to_2018Q4.csv.gz').as_posix()
rej = (DATA_DIR / 'rejected_2007_to_2018Q4.csv.gz').as_posix()
con.execute(f"CREATE TABLE IF NOT EXISTS accepted AS SELECT * FROM read_csv_auto('{acc}', sample_size=-1)")
con.execute(f"CREATE TABLE IF NOT EXISTS rejected AS SELECT * FROM read_csv_auto('{rej}', sample_size=-1)")

def run(sql):
    """Run SQL; return a DataFrame for SELECTs, None for DDL (CREATE/etc.)."""
    rel = con.execute(sql)
    return rel.df() if rel.description else None

print('Connected to', DATA_DIR / 'credit.duckdb')

## Phase 1.5 - Profile the accepted table

In [ ]:
run("""
-- rows in the table
SELECT COUNT(*) AS rows FROM accepted;
""")

In [ ]:
run("""
-- column names + DuckDB-inferred types
DESCRIBE accepted;
""")

In [ ]:
run("""
-- eyeball the first rows
SELECT * FROM accepted LIMIT 5;
""")

In [ ]:
run("""
-- target distribution (the raw material for Phase 3)
SELECT loan_status, COUNT(*) AS n
FROM accepted
GROUP BY loan_status
ORDER BY n DESC;
""")

In [ ]:
run("""
-- junk rows: real loans always have a loan_amnt
SELECT COUNT(*) AS junk_rows FROM accepted WHERE loan_amnt IS NULL;
""")

## Phase 1.6 - Inspect the rejected table (column names have spaces -> use double quotes)

In [ ]:
run("""
SELECT COUNT(*) AS rows FROM rejected;
""")

In [ ]:
run("""
-- SUMMARIZE = DuckDB's describe: min/max/avg/nulls per column
SUMMARIZE rejected;
""")

In [ ]:
run("""
SELECT * FROM rejected LIMIT 5;
""")

In [ ]:
run("""
-- Policy Code should be a single constant value (useless predictor)
SELECT "Policy Code", COUNT(*) AS n
FROM rejected
GROUP BY "Policy Code";
""")

## Phase 2 - Leakage-safe feature view (SELECT * EXCLUDE drops the 53 banned columns)

In [ ]:
run("""
CREATE OR REPLACE VIEW accepted_features AS
SELECT * EXCLUDE (
    "loan_status",
    "issue_d",
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "pymnt_plan",
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "deferral_term",
    "hardship_amount",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "hardship_length",
    "hardship_dpd",
    "hardship_loan_status",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term",
    "grade",
    "sub_grade",
    "int_rate",
    "installment",
    "id",
    "member_id",
    "url",
    "policy_code",
    "funded_amnt",
    "funded_amnt_inv",
    "emp_title",
    "title",
    "desc"
)
FROM accepted;
""")

In [ ]:
run("""
-- confirm 98 allowed columns remain
SELECT COUNT(*) AS allowed_cols
FROM information_schema.columns
WHERE table_name = 'accepted_features';
""")

## Phase 3 - Define the binary target on RESOLVED loans only

In [ ]:
run("""
CREATE OR REPLACE VIEW model_data AS
SELECT *,
  CASE WHEN loan_status IN ('Charged Off','Default') THEN 1
       WHEN loan_status = 'Fully Paid'              THEN 0 END AS target
FROM accepted
WHERE loan_status IN ('Fully Paid','Charged Off','Default');
""")

In [ ]:
run("""
-- class balance
SELECT target, COUNT(*) AS n FROM model_data GROUP BY target ORDER BY target;
""")

In [ ]:
run("""
-- cohort size + default rate
SELECT COUNT(*) AS rows, ROUND(AVG(target), 4) AS default_rate FROM model_data;
""")